# Rotary Embedding (RoPE) - Equations, Intuition, and PyTorch Code

RoPE encodes position by rotating each query/key feature pair in 2D, instead of adding a positional embedding vector. The main benefit is that relative position information appears naturally inside the attention dot product.

## What this notebook covers
- the core RoPE equations
- why the rotation is pairwise and why `head_dim` must be even
- a minimal PyTorch implementation
- a small self-attention module that applies RoPE to queries and keys
- a numerical check showing that the attention score depends on relative position

## 1. RoPE math

Assume one attention head has dimension $d_h$, and $d_h$ is even. RoPE groups the features into pairs:

$$(x_0, x_1), (x_2, x_3), \ldots, (x_{d_h-2}, x_{d_h-1})$$

Each pair is rotated by a position-dependent angle. For pair index $j \in \{0, \ldots, d_h/2 - 1\}$, define the angular frequency

$$\omega_j = \text{base}^{-2j / d_h}$$

and the angle at position $m$

$$\theta_{m,j} = m \cdot \omega_j$$

Then the pair $(x_{2j}, x_{2j+1})$ becomes

$$
\begin{bmatrix}
x'_{2j} \\
x'_{2j+1}
\end{bmatrix}
=
\begin{bmatrix}
\cos \theta_{m,j} & -\sin \theta_{m,j} \\
\sin \theta_{m,j} & \cos \theta_{m,j}
\end{bmatrix}
\begin{bmatrix}
x_{2j} \\
x_{2j+1}
\end{bmatrix}
$$

So in component form:

$$x'_{2j} = x_{2j}\cos\theta_{m,j} - x_{2j+1}\sin\theta_{m,j}$$

$$x'_{2j+1} = x_{2j}\sin\theta_{m,j} + x_{2j+1}\cos\theta_{m,j}$$

In code, this is usually written as

$$\mathrm{RoPE}(x_m) = x_m \odot \cos(\Theta_m) + \mathrm{rotate\_half}(x_m) \odot \sin(\Theta_m)$$

where `rotate_half` turns

$$(x_0, x_1, x_2, x_3, \ldots)$$

into

$$(-x_1, x_0, -x_3, x_2, \ldots)$$

## Why RoPE is useful in attention

If $R_m$ is the rotation applied at position $m$, then for query $q$ at position $m$ and key $k$ at position $n$:

$$\langle R_m q, R_n k \rangle = q^\top R_{n-m} k$$

The attention score depends on the relative offset $(n-m)$, not only the absolute positions. That is the main reason RoPE works well for transformers.

## 2. Imports

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## 3. Minimal RoPE helpers

The cache stores one cosine row and one sine row per sequence position. Each row has shape `(head_dim,)`.

The key implementation trick is to compute frequencies only for half the dimensions, then repeat each frequency twice so that every 2D pair shares one rotation angle.

In [2]:
def build_rope_cache(seq_len: int, head_dim: int, base: float = 10000.0, device=None, dtype=torch.float32):
    if head_dim % 2 != 0:
        raise ValueError("head_dim must be even for RoPE")

    half_dim = head_dim // 2
    inv_freq = 1.0 / (base ** (torch.arange(0, half_dim, device=device, dtype=torch.float32) / half_dim))
    positions = torch.arange(seq_len, device=device, dtype=torch.float32)
    freqs = torch.outer(positions, inv_freq)
    emb = torch.repeat_interleave(freqs, repeats=2, dim=-1)

    cos = emb.cos().to(dtype=dtype)
    sin = emb.sin().to(dtype=dtype)
    return cos, sin


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(start_dim=-2)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    while cos.ndim < x.ndim:
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)
    return x * cos + rotate_half(x) * sin

## 4. Toy example: rotating the same vector at different positions

Below, the same content vector is placed at several positions. The values change because the rotation angle changes with position, but the vector norm is preserved because every step is a pure rotation.

In [3]:
seq_len = 4
head_dim = 4
base_vector = torch.tensor([1.0, 0.0, 0.5, -0.5])

cos, sin = build_rope_cache(seq_len=seq_len, head_dim=head_dim)
toy = base_vector.view(1, 1, 1, head_dim).repeat(1, 1, seq_len, 1)
rotated = apply_rope(toy, cos, sin)

print("original vector:")
print(base_vector)
print()
for pos in range(seq_len):
    print(f"position {pos}: {rotated[0, 0, pos]}")

original_norm = base_vector.norm()
rotated_norms = rotated[0, 0].norm(dim=-1)
print()
print("original norm:", original_norm.item())
print("rotated norms:", rotated_norms)

original vector:
tensor([ 1.0000,  0.0000,  0.5000, -0.5000])

position 0: tensor([ 1.0000,  0.0000,  0.5000, -0.5000])
position 1: tensor([ 0.5403,  0.8415,  0.5050, -0.4950])
position 2: tensor([-0.4161,  0.9093,  0.5099, -0.4899])
position 3: tensor([-0.9900,  0.1411,  0.5148, -0.4848])

original norm: 1.2247449159622192
rotated norms: tensor([1.2247, 1.2247, 1.2247, 1.2247])


## 5. RoPE inside self-attention

In practice, RoPE is applied to queries and keys after projection and after reshaping into heads. Values are usually left unchanged.

The implementation below mirrors the Python script in this folder:
- project to Q, K, V
- reshape to `(batch, heads, seq_len, head_dim)`
- build or reuse cosine/sine caches
- rotate Q and K
- compute scaled dot-product attention

In [4]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, base: float = 10000.0):
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even")
        self.head_dim = head_dim
        self.base = base
        self.register_buffer("cos_cached", None, persistent=False)
        self.register_buffer("sin_cached", None, persistent=False)

    def _build_cache(self, seq_len: int, device: torch.device, dtype: torch.dtype):
        cos, sin = build_rope_cache(seq_len=seq_len, head_dim=self.head_dim, base=self.base, device=device, dtype=dtype)
        self.cos_cached = cos
        self.sin_cached = sin

    def forward(self, x: torch.Tensor):
        seq_len = x.size(-2)
        if (
            self.cos_cached is None
            or self.sin_cached is None
            or self.cos_cached.size(0) < seq_len
            or self.cos_cached.device != x.device
            or self.cos_cached.dtype != x.dtype
        ):
            self._build_cache(seq_len, x.device, x.dtype)
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


class RopeSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor | None = None, return_attn: bool = False):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(q)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if attn_mask is not None:
            scores = scores.masked_fill(~attn_mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        y = torch.matmul(attn, v)
        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.out(y)

        if return_attn:
            return output, attn
        return output


def make_causal_mask(seq_len: int, device=None) -> torch.Tensor:
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))
    return mask.unsqueeze(0).unsqueeze(0)

## 6. Run the attention module

This is the same usage pattern as a normal self-attention layer, except that Q and K get rotated before the score matrix is formed.

In [5]:
batch_size = 2
seq_len = 5
d_model = 8
n_heads = 2

x = torch.randn(batch_size, seq_len, d_model)
attn_mask = make_causal_mask(seq_len, x.device)

model = RopeSelfAttention(d_model=d_model, n_heads=n_heads)
output, attn = model(x, attn_mask=attn_mask, return_attn=True)

print("output shape:", output.shape)
print("attention shape:", attn.shape)
print()
print("attention weights for batch 0, head 0:")
print(attn[0, 0])

output shape: torch.Size([2, 5, 8])
attention shape: torch.Size([2, 2, 5, 5])

attention weights for batch 0, head 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3835, 0.6165, 0.0000, 0.0000, 0.0000],
        [0.3031, 0.3821, 0.3147, 0.0000, 0.0000],
        [0.2646, 0.3260, 0.1528, 0.2566, 0.0000],
        [0.1967, 0.1660, 0.2513, 0.1807, 0.2053]], grad_fn=<SelectBackward0>)


## 7. Numerical check: the score depends on relative position

If RoPE is doing what the math says, then entries with the same offset `(n - m)` should match up to small floating-point error. That means each diagonal of the score matrix should be nearly constant.

In [6]:
seq_len = 6
head_dim = 4
base_q = torch.tensor([[[[1.0, 2.0, -1.0, 0.5]]]])
base_k = torch.tensor([[[[0.5, -1.0, 1.5, 2.0]]]])
cos, sin = build_rope_cache(seq_len=seq_len, head_dim=head_dim)

scores = torch.empty(seq_len, seq_len)
for m in range(seq_len):
    q_m = apply_rope(base_q, cos[m:m + 1], sin[m:m + 1])[0, 0, 0]
    for n in range(seq_len):
        k_n = apply_rope(base_k, cos[n:n + 1], sin[n:n + 1])[0, 0, 0]
        scores[m, n] = torch.dot(q_m, k_n)

print("RoPE score matrix:")
print(scores)
print()
for offset in range(-(seq_len - 1), seq_len):
    diagonal = scores.diagonal(offset=offset)
    spread = (diagonal.max() - diagonal.min()).abs().item()
    print(f"offset {offset:>2}: values={diagonal.tolist()} spread={spread:.8f}")

RoPE score matrix:
tensor([[-2.0000,  0.4000,  1.9979,  1.3499, -0.9228, -2.7053],
        [-3.0209, -2.0000,  0.4000,  1.9979,  1.3499, -0.9228],
        [-1.7493, -3.0209, -2.0000,  0.4000,  1.9979,  1.3499],
        [ 0.6205, -1.7493, -3.0209, -2.0000,  0.4000,  1.9979],
        [ 1.8845,  0.6205, -1.7493, -3.0209, -2.0000,  0.4000],
        [ 0.8555,  1.8845,  0.6205, -1.7493, -3.0209, -2.0000]])

offset -5: values=[0.8555375337600708] spread=0.00000000
offset -4: values=[1.8844997882843018, 1.8844997882843018] spread=0.00000000
offset -3: values=[0.6204860806465149, 0.6204861998558044, 0.6204860806465149] spread=0.00000012
offset -2: values=[-1.7492709159851074, -1.7492709159851074, -1.7492709159851074, -1.7492709159851074] spread=0.00000000
offset -1: values=[-3.020869731903076, -3.0208699703216553, -3.0208699703216553, -3.020869731903076, -3.0208702087402344] spread=0.00000048
offset  0: values=[-2.0, -2.0, -2.0, -2.0, -1.999999761581421, -2.000000238418579] spread=0.00000048
of